# Querying the Victoria Urban Planning DuckDB Database

This notebook serves as a guide for developers and analysts to explore the analytical data compiled by the Stream 2 (Traffic Volumes & Parking) ingestion pipeline.

The database is stored at:
`team_b/data/parking_analytics.duckdb`

## Table Schemas & Description

1. **`blocks_summary`**: High-level comparison per street block, showing parking capacity before/after and average occupancy rates.
   - `suburb` (VARCHAR)
   - `street_name` (VARCHAR)
   - `block_desc` (VARCHAR)
   - `geom_json` (VARCHAR - GeoJSON line geometry)
   - `baseline_bays` (INTEGER - Parking capacity before intervention)
   - `final_bays` (INTEGER - Parking capacity after intervention)
   - `bays_removed` (INTEGER - Net loss of parking bays)
   - `pre_occupancy` (DOUBLE - Average baseline occupancy rate, 0.0 to 1.0)
   - `post_occupancy` (DOUBLE - Average post-intervention occupancy rate, 0.0 to 1.0)

2. **`hourly_occupancy`**: Granular hourly-level parking metrics.
   - `year` (INTEGER - 2013 for baseline, 2014 for post-intervention)
   - `suburb` (VARCHAR)
   - `street_name` (VARCHAR)
   - `block_desc` (VARCHAR)
   - `hr` (TIMESTAMP - Hour bucket start time)
   - `occupied_minutes` (DOUBLE)
   - `total_minutes` (DOUBLE)
   - `occupancy_rate` (DOUBLE - 0.0 to 1.0)
   - `bay_count` (INTEGER - Active capacity during this month)

3. **`traffic_volumes`**: Hourly intersection traffic logs.
   - `timestamp` (TIMESTAMP - 15-minute interval start)
   - `intersection_id` (INTEGER)
   - `traffic_volume` (INTEGER - Vehicle count)
   - `degree_of_saturation` (DOUBLE - Ratio of volume to capacity)

4. **`block_geometries`**: Lookup table containing dissolved geometries for each block.
   - `suburb`, `street_name`, `block_desc` (VARCHAR)
   - `geom_json` (VARCHAR - Dissolved GeoJSON LineString)

In [1]:
import duckdb
import pandas as pd

# Connect to the DuckDB database (read-only mode)
DB_PATH = 'data/parking_analytics.duckdb'
conn = duckdb.connect(DB_PATH, read_only=True)

# Set pandas options for better layout
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.width', 1000)

print("Successfully connected to DuckDB!")

Successfully connected to DuckDB!


## 1. Overview of Interventions by Suburb

Let's begin by checking which suburbs have the highest count of intervention street blocks present in our database. We can query the `blocks_summary` table.

In [2]:
query_blocks = """
SELECT 
    suburb, 
    COUNT(*) AS num_blocks
FROM blocks_summary
GROUP BY suburb
ORDER BY num_blocks DESC
"""
df_blocks = conn.execute(query_blocks).df()
df_blocks

,suburb,num_blocks
0,North Melbourne,90
1,Carlton,82
2,Melbourne (CBD),48
3,East Melbourne,37
4,Parkville,37
5,Kensington,36
6,West Melbourne (Residential),32
7,Docklands,26
8,Melbourne (Remainder),20
9,Southbank,10


## 2. Parking Capacity Impact

Streetscape interventions such as bike lane installations often require removing on-street parking spaces. Let's see the total number of baseline parking bays vs final parking bays, and analyze which suburbs lost the most bays.

In [3]:
query_capacity = """
SELECT 
    suburb,
    SUM(baseline_bays) AS baseline_bays,
    SUM(final_bays) AS final_bays,
    SUM(bays_removed) AS total_bays_removed,
    ROUND(SUM(bays_removed) * 100.0 / SUM(baseline_bays), 2) AS pct_removed
FROM blocks_summary
GROUP BY suburb
ORDER BY total_bays_removed DESC
"""
df_capacity = conn.execute(query_capacity).df()
df_capacity

,suburb,baseline_bays,final_bays,total_bays_removed,pct_removed
0,Melbourne (CBD),528.0,523.0,5.0,0.95
1,West Melbourne (Residential),407.0,404.0,3.0,0.74
2,Port Melbourne,263.0,262.0,1.0,0.38
3,Docklands,395.0,397.0,-2.0,-0.51
4,South Yarra,44.0,47.0,-3.0,-6.82
5,West Melbourne (Industrial),121.0,129.0,-8.0,-6.61
6,Southbank,208.0,220.0,-12.0,-5.77
7,Kensington,584.0,600.0,-16.0,-2.74
8,North Melbourne,1348.0,1369.0,-21.0,-1.56
9,East Melbourne,582.0,608.0,-26.0,-4.47


## 3. Occupancy Rate Comparison

Let's look at which street blocks saw the biggest increase or decrease in parking occupancy rates between the baseline and post-intervention periods. A higher post-occupancy rate could suggest parking stress or displacement, while a lower rate could mean reduced parking demand.

In [4]:
query_occupancy_change = """
SELECT 
    street_name,
    block_desc,
    suburb,
    baseline_bays,
    final_bays,
    ROUND(pre_occupancy, 4) AS pre_occupancy,
    ROUND(post_occupancy, 4) AS post_occupancy,
    ROUND(post_occupancy - pre_occupancy, 4) AS occupancy_diff
FROM blocks_summary
WHERE pre_occupancy IS NOT NULL AND post_occupancy IS NOT NULL
ORDER BY ABS(occupancy_diff) DESC
LIMIT 10
"""
df_occupancy_change = conn.execute(query_occupancy_change).df()
df_occupancy_change

,street_name,block_desc,suburb,baseline_bays,final_bays,pre_occupancy,post_occupancy,occupancy_diff
0,Curzon Street,Curzon Street between Provost Street and Queen...,North Melbourne,1,1,0.2108,0.5701,0.3593
1,La Trobe Street,La Trobe Street between Adderley Street and Wu...,Docklands,1,1,0.1928,0.5388,0.3460
2,Adderley Street,Adderley Street between Dudley Street and Ross...,West Melbourne (Residential),1,2,0.6639,0.3425,-0.3215
3,Courtney Street,Courtney Street between Villiers Street and Ha...,North Melbourne,2,5,0.3847,0.1060,-0.2787
4,Moray Street,Moray Street between City Road and Catherine S...,Southbank,3,1,0.2433,0.4771,0.2337
5,Exhibition Street,Exhibition Street between Flinders Street and ...,Melbourne (CBD),1,1,0.3733,0.5740,0.2007
6,Adderley Street,Adderley Street between Roden Street and Hawke...,West Melbourne (Residential),5,5,0.2736,0.1037,-0.1699
7,Errol Street,Errol Street between Chapman Street and Harker...,North Melbourne,1,1,0.4750,0.6383,0.1633
8,La Trobe Street,La Trobe Street between Victoria Street and Ex...,Melbourne (CBD),2,4,0.3264,0.1670,-0.1594
9,Swanston Street,Swanston Street between La Trobe Street and Li...,Melbourne (CBD),2,2,0.2753,0.4274,0.1520


## 4. Temporal (Hour-of-Day) Occupancy Profiles

To inspect how the parking pattern changes over a typical 24-hour day, we can analyze the `hourly_occupancy` table. Let's look at one specific high-impact street block (e.g., `ADDERLEY STREET`) and retrieve the average occupancy rate per hour for the baseline year (2013) vs the post-intervention year (2014).

In [5]:
query_temporal = """
SELECT 
    year,
    EXTRACT(HOUR FROM hr) AS hour_of_day,
    AVG(occupancy_rate) AS avg_occupancy,
    AVG(bay_count) AS avg_capacity
FROM hourly_occupancy
WHERE street_name = 'ADDERLEY STREET'
GROUP BY year, hour_of_day
ORDER BY year, hour_of_day
"""
df_temporal = conn.execute(query_temporal).df()
df_pivot = df_temporal.pivot(index='hour_of_day', columns='year', values='avg_occupancy')
df_pivot

year
hour_of_day


## 5. Traffic Volume & Intersection Saturation

To see the traffic flow patterns near the interventions, we can analyze the `traffic_volumes` table (populated by SCATS telemetry data). Let's calculate the average hourly traffic volume and degree of saturation across all intersections in the dataset.

In [6]:
query_traffic = """
SELECT 
    EXTRACT(HOUR FROM timestamp) AS hour_of_day,
    ROUND(AVG(traffic_volume), 2) AS avg_vehicle_count,
    ROUND(AVG(degree_of_saturation), 4) AS avg_saturation
FROM traffic_volumes
GROUP BY hour_of_day
ORDER BY hour_of_day
"""
df_traffic = conn.execute(query_traffic).df()
df_traffic

,hour_of_day,avg_vehicle_count,avg_saturation
0,0,99.57,0.1082
1,1,101.00,0.1085
2,2,123.75,0.1275
3,3,198.31,0.1988
4,4,323.26,0.3238
5,5,447.26,0.4469
6,6,499.77,0.5001
7,7,447.47,0.4476
8,8,324.54,0.3255
9,9,199.17,0.1991


## 6. Spatial Geometries

For mapping and GIS integrations, block geometries are stored in the database in raw GeoJSON format (`geom_json`). Let's retrieve a few block descriptions and a preview of their geometries to show how frontend clients can query this data for rendering.

In [7]:
query_geom = """
SELECT 
    suburb,
    street_name,
    block_desc,
    SUBSTR(geom_json, 1, 120) || ' ...' AS geom_geojson_preview
FROM block_geometries
LIMIT 5
"""
df_geom = conn.execute(query_geom).df()
df_geom

,suburb,street_name,block_desc,geom_geojson_preview
0,Carlton,Barkly Street,Barkly Street between Canning Street and Rathd...,"{""type"": ""MultiLineString"", ""coordinates"": [[[..."
1,Carlton,Bouverie Street,Bouverie Street between Lincoln Square North a...,"{""type"": ""MultiLineString"", ""coordinates"": [[[..."
2,Carlton,Bouverie Street,Bouverie Street between Lincoln Square South a...,"{""type"": ""MultiLineString"", ""coordinates"": [[[..."
3,Carlton,Bouverie Street,Bouverie Street between Pelham Street and Linc...,"{""type"": ""MultiLineString"", ""coordinates"": [[[..."
4,Carlton,Bouverie Street,Bouverie Street between Queensberry Street and...,"{""type"": ""MultiLineString"", ""coordinates"": [[[..."


In [8]:
# Cleanly close the database connection
conn.close()
print("Connection closed successfully.")

Connection closed successfully.
